# Mục 3f — Hai ma trận cho ba hồ sơ

**Võ Minh Huy — 2311110117**

Tương ứng file SQL 12_ma_tran_3_ho_so.sql. Dùng sáu chỉ số định lượng: tuổi, CPK, EF, platelets đã làm sạch, creatinine và natri. Không dùng ID, nhãn, mã danh mục hay hai nhánh làm mịn làm tọa độ. Pearson ở đây so sánh **hai vector hồ sơ**, không phải tương quan giữa các thuộc tính trên toàn bộ bệnh nhân.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

thu_muc = Path.cwd()
if thu_muc.name == "python":
    thu_muc = thu_muc.parent
du_lieu = pd.read_csv(thu_muc / "VoMinhHuy_Heart Failure Prediction_after_Python.csv")
cot = ["age", "creatinine_phosphokinase", "ejection_fraction",
       "platelets", "serum_creatinine", "serum_sodium"]
if len(du_lieu) < 3 or du_lieu[cot].isna().any().any():
    raise ValueError("Cần ít nhất ba hồ sơ và đủ sáu giá trị.")
if not du_lieu["MaHoSo"].is_unique:
    raise ValueError("Khóa nội bộ bị trùng.")
nhan = ["A", "B", "C"]
chon = du_lieu.sort_values(cot + ["MaHoSo"]).head(3).copy()
chon.insert(0, "HoSo", nhan)
ten_ngan = {"creatinine_phosphokinase":"CPK", "ejection_fraction":"EF",
            "serum_creatinine":"creatinine", "serum_sodium":"sodium"}
ho_so = chon[["HoSo", "MaHoSo"] + cot].rename(columns=ten_ngan)
ho_so

,HoSo,MaHoSo,age,CPK,EF,platelets,creatinine,sodium
242,A,243,40.0,90.0,35.0,255000.0,1.1,136.0
206,B,207,40.0,101.0,40.0,226000.0,0.8,141.0
232,C,233,40.0,129.0,35.0,255000.0,0.9,137.0


## 1. Chuẩn hóa theo toàn bộ dữ liệu

Mean và độ lệch chuẩn mẫu tính từ toàn bộ 299 hồ sơ: z = (x − mean) / std, với std dùng n−1. Giữ nguyên độ chính xác khi tính, chỉ làm tròn bảng hiển thị. A, B, C được chọn theo thứ tự sáu giá trị, nên đối chiếu bằng chỉ số gốc chứ không bằng MaHoSo giữa hai môi trường.

In [2]:
mean = du_lieu[cot].mean()
std = du_lieu[cot].std(ddof=1)
if (std <= 0).any() or std.isna().any():
    raise ValueError("Không chuẩn hóa được thuộc tính có std bằng 0.")
tham_so = pd.DataFrame({"ThuocTinh":cot,
    "SoMau":du_lieu[cot].count().to_numpy(), "Mean":mean.to_numpy(), "StdMau":std.to_numpy()})
tham_so.round(6)

,ThuocTinh,SoMau,Mean,StdMau
0,age,299,60.742923,11.250608
1,creatinine_phosphokinase,299,552.712375,958.093295
2,ejection_fraction,299,38.083612,11.834841
3,platelets,299,263868.722341,94807.769902
4,serum_creatinine,299,1.401204,1.030011
5,serum_sodium,299,136.448161,4.300752


In [3]:
z = (chon[cot] - mean) / std
z.index = pd.Index(nhan, name="HoSo")
z_hien_thi = z.rename(columns=ten_ngan)
z_hien_thi.round(6)

,age,CPK,EF,platelets,creatinine,sodium
HoSo,,,,,,
A,-1.843716,-0.482951,-0.260554,-0.093544,-0.292428,-0.104205
B,-1.843716,-0.471470,0.161928,-0.399426,-0.583687,1.058382
C,-1.843716,-0.442245,-0.260554,-0.093544,-0.486601,0.128312


## 2. Ma trận tương quan Pearson giữa ba hồ sơ

Mỗi hàng là một vector gồm sáu giá trị Z. np.corrcoef mặc định rowvar=True để tính tương quan giữa hàng với hàng. Pearson đo sự tương đồng về hình dạng sau khi trừ trung bình riêng của mỗi vector. Không dùng các số này để kết luận quan hệ nhân quả hay nguy cơ bệnh.

In [4]:
if (z.std(axis=1, ddof=1) < 1e-12).any():
    raise ValueError("Vector gần như hằng, không tính được Pearson.")
tuong_quan = pd.DataFrame(np.corrcoef(z.to_numpy(), rowvar=True), index=nhan, columns=nhan)
tuong_quan.index.name = "HoSo"
tuong_quan.round(6)

,A,B,C
HoSo,,,
A,1.000000,0.822155,0.981048
B,0.822155,1.000000,0.898330
C,0.981048,0.898330,1.000000


## 3. Ma trận khác biệt Euclidean

d(i,j) = căn của tổng bình phương chênh lệch ở sáu tọa độ Z. Khoảng cách nhỏ hơn nghĩa là gần nhau hơn trên sáu chỉ số đã chọn. Khoảng cách không phải xác suất hoặc mức độ bệnh.

In [5]:
gia_tri_z = z.to_numpy()
khoang_cach = np.sqrt(((gia_tri_z[:, None, :] - gia_tri_z[None, :, :]) ** 2).sum(axis=2))
khac_biet = pd.DataFrame(khoang_cach, index=nhan, columns=nhan)
khac_biet.index.name = "HoSo"
khac_biet.round(6)

,A,B,C
HoSo,,,
A,0.000000,1.307145,0.305654
B,1.307145,0.000000,1.071151
C,0.305654,1.071151,0.000000


## 4. Kiểm tra, đối chiếu và lưu kết quả

Kiểm tra đối xứng, miền Pearson từ −1 đến 1, khoảng cách không âm và đường chéo. Các tham số SQL có thể khác rất nhỏ vì bảng import float; báo chênh lệch thực tế, không ép hai kết quả bằng nhau. Dùng cùng ba tổ hợp chỉ số gốc để đối chiếu.

In [6]:
assert np.allclose(tuong_quan, tuong_quan.T)
assert np.allclose(khac_biet, khac_biet.T)
assert np.allclose(np.diag(tuong_quan), 1)
assert np.allclose(np.diag(khac_biet), 0)
assert np.abs(tuong_quan.to_numpy()).max() <= 1 + 1e-12
assert khac_biet.to_numpy().min() >= 0

thu_muc_kq = thu_muc / "ket_qua/20_ma_tran_3_ho_so"
thu_muc_kq.mkdir(parents=True, exist_ok=True)
sql_ho_so = pd.read_csv(thu_muc_kq / "ho_so_sql.csv")
cot_ngan = [ten_ngan.get(c, c) for c in cot]
assert sql_ho_so["HoSo"].tolist() == nhan
ho_so_khop = np.allclose(ho_so[cot_ngan].to_numpy(), sql_ho_so[cot_ngan].to_numpy(), rtol=0, atol=1e-6)
if not ho_so_khop:
    raise ValueError("Ba hồ sơ SQL/Python khác giá trị; cần kiểm tra cách chọn.")
sql_corr = pd.read_csv(thu_muc_kq / "tuong_quan_sql.csv", index_col="HoSo").loc[nhan, nhan]
sql_dist = pd.read_csv(thu_muc_kq / "khac_biet_sql.csv", index_col="HoSo").loc[nhan, nhan]
chenh_corr = float(np.abs(tuong_quan.to_numpy() - sql_corr.to_numpy()).max())
chenh_dist = float(np.abs(khac_biet.to_numpy() - sql_dist.to_numpy()).max())
bang_kiem_tra = pd.DataFrame({"SoHoSo":[3], "SoThuocTinh":[6], "SoMauChuanHoa":[len(du_lieu)],
    "HoSoGocKhopSQL":[ho_so_khop], "DoiXungVaDuongCheo":[True],
    "ChenhPearsonSoVoiSQL6SoLe":[chenh_corr], "ChenhEuclideanSoVoiSQL6SoLe":[chenh_dist]})
for ten, bang, co_index in [("ho_so_python.csv", ho_so, False),
    ("tham_so_python.csv", tham_so, False), ("z_score_python.csv", z_hien_thi, True),
    ("tuong_quan_python.csv", tuong_quan, True), ("khac_biet_python.csv", khac_biet, True),
    ("kiem_tra_python.csv", bang_kiem_tra, False)]:
    bang.to_csv(thu_muc_kq / ten, index=co_index, encoding="utf-8-sig")
bang_kiem_tra

,SoHoSo,SoThuocTinh,SoMauChuanHoa,HoSoGocKhopSQL,DoiXungVaDuongCheo,ChenhPearsonSoVoiSQL6SoLe,ChenhEuclideanSoVoiSQL6SoLe
0,3,6,299,True,True,3.802306e-07,1.294158e-07


**Nhận xét:** So sánh ba cặp A–B, A–C và B–C. Cặp có Euclidean nhỏ nhất gần nhau nhất trên sáu tọa độ đã chuẩn hóa; Pearson lớn hơn phản ánh hình dạng tương đồng hơn. Ba hồ sơ chỉ là ví dụ theo yêu cầu đề bài, không đại diện cho toàn bộ bệnh nhân. Giữ nguyên các file after.